In [2]:
import os
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("GITHUB_TOKEN")
REPO = "mesale/gpt2-from-scratch"
!git clone https://{token}@github.com/{REPO}.git /kaggle/temp/repo
%cd /kaggle/temp/repo
!git remote set-url origin https://github.com/{REPO}.git
!pip install -q tokenizers pyyaml

Cloning into '/kaggle/temp/repo'...
remote: Enumerating objects: 182, done.
remote: Counting objects: 100% (182/182), done.
remote: Compressing objects: 100% (144/144), done.
remote: Total 182 (delta 68), reused 149 (delta 35), pack-reused 0 (from 0)
Receiving objects: 100% (182/182), 454.00 KiB | 5.22 MiB/s, done.
Resolving deltas: 100% (68/68), done.
/kaggle/temp/repo


In [3]:
import glob
bins = glob.glob("/kaggle/input/**/tinystories_train.bin", recursive=True)
assert bins, "tinystories_train.bin not found: is the dataset attached, and did data-prep finish?"
DATA_DIR = os.path.dirname(bins[0])
print(DATA_DIR, os.listdir(DATA_DIR))
n = os.path.getsize(bins[0]) // 2          # uint16 = 2 bytes per token
print(f"{n:,} train tokens | the run sees {8000 * 32768:,} tokens = {8000 * 32768 / n:.2f} epochs")

/kaggle/input/datasets/mesaletsegaye/tinystories-v2-bpe4096/data ['shk_train.bin', 'shk_val.bin', 'tinystories_train.bin', 'tinystories_val.bin']
552,529,554 train tokens | the run sees 262,144,000 tokens = 0.47 epochs


In [3]:
for h in [6, 1, 2, 12]:
    !python -m src.train --config configs/ablation_heads.yaml --set model.n_head={h} train.run_name=heads_{h} train.data_dir={DATA_DIR} train.out_dir=/kaggle/working/runs

12,318,720 params | 32,768 tokens/step | device=cuda amp=True
[eval] step 0: train 8.3959 | val 8.3969
step     50 | loss 6.1341 | lr 5.00e-04 | gnorm 0.83 | 8,984 tok/s
step    100 | loss 4.0388 | lr 1.00e-03 | gnorm 0.86 | 15,025 tok/s
step    150 | loss 3.4891 | lr 9.97e-04 | gnorm 0.72 | 21,775 tok/s
step    200 | loss 3.0644 | lr 9.89e-04 | gnorm 0.85 | 28,478 tok/s
step    250 | loss 2.8066 | lr 9.75e-04 | gnorm 0.76 | 33,665 tok/s
[eval] step 250: train 2.6898 | val 2.7018
step    300 | loss 2.6060 | lr 9.56e-04 | gnorm 0.72 | 38,771 tok/s
step    350 | loss 2.4737 | lr 9.32e-04 | gnorm 0.67 | 43,907 tok/s
step    400 | loss 2.3626 | lr 9.02e-04 | gnorm 0.73 | 49,034 tok/s
step    450 | loss 2.2793 | lr 8.69e-04 | gnorm 0.69 | 54,015 tok/s
step    500 | loss 2.1992 | lr 8.31e-04 | gnorm 0.66 | 56,593 tok/s
[eval] step 500: train 2.1605 | val 2.1658
step    550 | loss 2.1404 | lr 7.90e-04 | gnorm 0.60 | 57,065 tok/s
step    600 | loss 2.0811 | lr 7.46e-04 | gnorm 0.61 | 62,258 to

In [4]:
!python -m src.train --config configs/ablation_heads.yaml --set model.n_head=6 train.seed=1338 train.run_name=heads_6_seed2 train.data_dir={DATA_DIR} train.out_dir=/kaggle/working/runs

12,318,720 params | 32,768 tokens/step | device=cuda amp=True
[eval] step 0: train 8.4576 | val 8.4589
step     50 | loss 6.1518 | lr 5.00e-04 | gnorm 0.82 | 70,961 tok/s
step    100 | loss 4.0272 | lr 1.00e-03 | gnorm 0.89 | 71,131 tok/s
step    150 | loss 3.4875 | lr 9.97e-04 | gnorm 0.97 | 73,152 tok/s
step    200 | loss 3.0626 | lr 9.89e-04 | gnorm 1.13 | 73,474 tok/s
step    250 | loss 2.7852 | lr 9.75e-04 | gnorm 0.79 | 72,585 tok/s
[eval] step 250: train 2.6768 | val 2.6888
step    300 | loss 2.6038 | lr 9.56e-04 | gnorm 0.66 | 72,480 tok/s
step    350 | loss 2.4613 | lr 9.32e-04 | gnorm 0.68 | 72,892 tok/s
step    400 | loss 2.3531 | lr 9.02e-04 | gnorm 0.69 | 73,190 tok/s
step    450 | loss 2.2543 | lr 8.69e-04 | gnorm 0.63 | 73,245 tok/s
step    500 | loss 2.2008 | lr 8.31e-04 | gnorm 0.61 | 72,996 tok/s
[eval] step 500: train 2.1538 | val 2.1611
step    550 | loss 2.1363 | lr 7.90e-04 | gnorm 0.65 | 72,592 tok/s
step    600 | loss 2.0726 | lr 7.46e-04 | gnorm 0.58 | 73,358 t

In [6]:
import os, shutil, yaml, pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

root, res = "/kaggle/working/runs", "/kaggle/temp/repo/results/ablation_heads"
os.makedirs(res, exist_ok=True)
names = [n for n in ["heads_1", "heads_2", "heads_6", "heads_12", "heads_6_seed2"] if os.path.isdir(f"{root}/{n}")]

fig, ax = plt.subplots(figsize=(7, 4))
rows = []
for n in names:
    ev = pd.read_csv(f"{root}/{n}/eval_log.csv")
    tr = pd.read_csv(f"{root}/{n}/train_log.csv")
    mc = yaml.safe_load(open(f"{root}/{n}/config.yaml"))["model"]
    late = ev[ev.step >= 500]
    ax.plot(late.step, late.val_loss, "o-", label=n)
    rows.append({"run": n, "heads": mc["n_head"], "head_dim": mc["n_embd"] // mc["n_head"],
                 "final_val": ev.val_loss.iloc[-1], "best_val": ev.val_loss.min(),
                 "tok_per_s": tr.tokens_per_sec.iloc[1:].mean()})
    os.makedirs(f"{res}/{n}", exist_ok=True)
    for f in ["train_log.csv", "eval_log.csv", "config.yaml"]:
        shutil.copy(f"{root}/{n}/{f}", f"{res}/{n}")

ax.set_xlabel("step"); ax.set_ylabel("val loss (nats/token)"); ax.legend(); ax.grid(alpha=0.3)
fig.savefig(f"{res}/val_by_heads.png", dpi=150, bbox_inches="tight")
table = pd.DataFrame(rows)
table.to_csv(f"{res}/summary.csv", index=False)
print(table.to_string(index=False))

          run  heads  head_dim  final_val  best_val    tok_per_s
      heads_1      1       384   1.719831  1.719831 89596.380935
      heads_2      2       192   1.711889  1.711889 81853.160046
      heads_6      6        64   1.702770  1.702770 59406.347306
     heads_12     12        32   1.704621  1.704621 59004.390643
heads_6_seed2      6        64   1.698366  1.698366 71211.056969


In [5]:
import pandas as pd, glob
for f in sorted(glob.glob("results/ablation_heads/heads_*/train_log.csv")):
    t = pd.read_csv(f)
    print(f.split("/")[-2], f"{t.tokens_per_sec.median():,.0f} tok/s (median)")

heads_1 89,686 tok/s (median)
heads_12 59,013 tok/s (median)
heads_2 81,888 tok/s (median)
heads_6 66,076 tok/s (median)
heads_6_seed2 71,316 tok/s (median)


In [ ]:
%cd /kaggle/temp/repo
!git config user.email "mesalemtsegaye@gmail.com" && git config user.name "kaggle"
!git add results && git commit -m "Add head-count ablation results" && git push https://{token}@github.com/{REPO}.git HEAD:main